In [8]:
import pandas as pd
from functools import reduce

# --- 1. Setup: Define File Paths and Final Column Mappings for 2021 Data ---
file_paths = {
    'edu_2021': '/Users/callumhall/Documents/Dissertation/Datasets/Archive/Census Data/Archive/2021/Education 2021.csv',
    'tenure_2021': '/Users/callumhall/Documents/Dissertation/Datasets/Census Data/2021/Tenure 2021.csv',
    'occ_2021': '/Users/callumhall/Documents/Dissertation/Datasets/Archive/Census Data/Archive/2021/Occupation 2021.csv',
    'age_2021': '/Users/callumhall/Documents/Dissertation/Datasets/Archive/Census Data/Archive/2021/Age 2021.csv',
    'eth_2021': '/Users/callumhall/Documents/Dissertation/Datasets/Archive/Census Data/Archive/2021/Ethnic Group 2021.csv',
    'property_2021': '/Users/callumhall/Documents/Dissertation/Datasets/Archive/Census Data/Archive/2021/Property Type.csv'
}

In [12]:
# Renaming Columns
edu_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total: All usual residents aged 16 years and over': 'Total_Population', 'No qualifications': 'Edu_No_Quals', 'Level 1 and entry level qualifications': 'Edu_Level_1_Quals', 'Level 2 qualifications': 'Edu_Level_2_Quals', 'Apprenticeship': 'Edu_Apprentice', 'Level 3 qualifications': 'Edu_Level_3_Quals', 'Level 4 qualifications or above': 'Edu_Level_4_Plus'}
tenure_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total: All households': 'Total_Households', 'Owned': 'Tenure_Owned', 'Social rented': 'Tenure_Social_Rented','Private rented': 'Tenure_Private_Rented'}
occ_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total: All usual residents aged 16 years and over in employment the week before the census': 'Total_Employed', '1. Managers, directors and senior officials': 'Occ_Managers', '2. Professional occupations': 'Occ_Professional', '3. Associate professional and technical occupations': 'Occ_Associate_Professional', '5. Skilled trades occupations': 'Occ_Skilled_Trades'}
age_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total': 'Total_Age_Population','Aged 15 to 19 years': 'Age_15_19','Aged 20 to 24 years': 'Age_20_24', 'Aged 25 to 29 years': 'Age_25_29', 'Aged 30 to 34 years': 'Age_30_34', 'Aged 35 to 39 years': 'Age_35_39', 'Aged 40 to 44 years': 'Age_40_44', 'Aged 45 to 49 years': 'Age_45_49', 'Aged 50 to 54 years': 'Age_50_54', 'Aged 55 to 59 years': 'Age_55_59', 'Aged 60 to 64 years': 'Age_60_64'}
eth_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total: All usual residents': 'Total_Ethnic_Population', 'White': 'Ethnic_White', 'Mixed or Multiple ethnic groups': 'Ethnic_Mixed', 'Asian, Asian British or Asian Welsh': 'Ethnic_Asian', 'Black, Black British, Black Welsh, Caribbean or African': 'Ethnic_Black'}
property_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name','Total': 'Total_Properties', 'Detached': 'Property_Detached', 'Semi-detached': 'Property_Semi', 'Terraced': 'Property_Terraced', 'In a purpose-built block of flats or tenement': 'Flats'}

# Define the number of header rows to skip for each file
skip_values_21 = {'edu_2021': 7, 'tenure_2021': 7, 'occ_2021': 7, 'age_2021': 5, 'eth_2021': 7, 'property_2021': 7}

# --- 2. Reusable Cleaning Function ---
def load_and_clean_census_file(path, column_map, skiprows):
    try:
        df = pd.read_csv(path, skiprows=skiprows, usecols=list(column_map.keys()), encoding='ISO-8859-1')
        df = df.rename(columns=column_map)
        df[['LSOA_Code', 'LSOA_Name']] = df['LSOA_Code_Name'].str.split(' : ', n=1, expand=True)
        df = df.drop(columns=['LSOA_Code_Name', 'LSOA_Name'])
        df.dropna(subset=['LSOA_Code'], inplace=True)
        data_cols = [col for col in df.columns if col != 'LSOA_Code']
        for col in data_cols:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
        return df
    except Exception as e:
        print(f"Error processing file {path}: {e}")
        return pd.DataFrame()

# --- 3. Load and Clean Each 2021 File ---
print("Loading and cleaning 2021 data...")
df_edu = load_and_clean_census_file(file_paths['edu_2021'], edu_cols_21, skip_values_21['edu_2021'])
df_tenure = load_and_clean_census_file(file_paths['tenure_2021'], tenure_cols_21, skip_values_21['tenure_2021'])
df_occ = load_and_clean_census_file(file_paths['occ_2021'], occ_cols_21, skip_values_21['occ_2021'])
df_age = load_and_clean_census_file(file_paths['age_2021'], age_cols_21, skip_values_21['age_2021'])
df_eth = load_and_clean_census_file(file_paths['eth_2021'], eth_cols_21, skip_values_21['eth_2021'])
df_property = load_and_clean_census_file(file_paths['property_2021'], property_cols_21, skip_values_21['property_2021'])

# --- 4. Merge into a Single DataFrame ---
dfs_to_merge = [df for df in [df_edu, df_tenure, df_occ, df_age, df_eth, df_property] if not df.empty]

if len(dfs_to_merge) == 5:
    df_2021_final = reduce(lambda left, right: pd.merge(left, right, on='LSOA_Code', how='inner'), dfs_to_merge)
    print("\n--- Final Cleaned 2021 DataFrame ---")
    df_2021_final.info()
else:
    print("\nOne or more files failed to load. Merge was not performed.")

Loading and cleaning 2021 data...
Error processing file /Users/callumhall/Documents/Dissertation/Datasets/Census Data/2021/Tenure 2021.csv: [Errno 2] No such file or directory: '/Users/callumhall/Documents/Dissertation/Datasets/Census Data/2021/Tenure 2021.csv'

--- Final Cleaned 2021 DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4862 entries, 0 to 4861
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Total_Population            4862 non-null   int64  
 1   Edu_No_Quals                4862 non-null   float64
 2   Edu_Level_1_Quals           4862 non-null   float64
 3   Edu_Level_2_Quals           4862 non-null   float64
 4   Edu_Apprentice              4862 non-null   float64
 5   Edu_Level_3_Quals           4862 non-null   float64
 6   Edu_Level_4_Plus            4862 non-null   float64
 7   LSOA_Code                   4862 non-null   object 
 8   Total_Employed    

In [14]:
# Define the name for your new CSV file
output_path = 'cleaned_census_final.csv'

# Save the DataFrame to a CSV file
# index=False prevents pandas from writing the DataFrame index as a column
df_2021_final.to_csv(output_path, index=False)

print(f"DataFrame successfully saved to: {output_path}")

DataFrame successfully saved to: cleaned_census_final.csv
